In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
CONFIG = json.loads((TASK.parent/'config.json').read_text())
INPUT, OUTPUT = TASK/'input', TASK/'output'
FIRST, LAST = CONFIG['first_roster_year'], CONFIG['last_publication_year']
ROSTERS = ['department-faculty', 'program-faculty', 'medical-faculty']
RANKS = ['Assistant Professor', 'Associate Professor', 'Professor']
FIELD_IDS = {code: field['field_id'] for field in CONFIG['fields'] for code in field['aarc_ids']}

In [ ]:
def read_roster(name):
    path = INPUT/f'{name}.csv'
    columns = pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns
    year = next(c for c in ('AADYEAR', 'AADYear') if c in columns)
    department = next(c for c in ('UnitRegistryId', 'UnitId') if c in columns)
    required = ['PersonId', year, 'InstitutionId', department, 'RankTypeId',
                'TaxonomyLevel01Id', 'TaxonomyLevel01Name']
    try:
        frame = pd.read_csv(path, usecols=required, encoding='utf-8-sig', low_memory=False)
    except UnicodeDecodeError:
        frame = pd.read_csv(path, usecols=required, encoding='cp1252', low_memory=False)
    frame = frame.rename(columns={year:'Year', department:'DepartmentId'})
    for column in frame.columns.drop('TaxonomyLevel01Name'):
        frame[column] = pd.to_numeric(frame[column], errors='coerce').astype('Int64')
    return frame

In [ ]:
institutions = pd.read_csv(INPUT/'phd_granting_institutions.csv', usecols=['InstitutionId']).InstitutionId
rank_map = pd.read_csv(INPUT/'professorial_rank.csv').set_index('Id').Name
raw = pd.concat([read_roster(name) for name in ROSTERS], ignore_index=True)
raw['Rank'] = raw.RankTypeId.map(rank_map)

In [ ]:
# match the prior prep's employing institutions, retained ranks, and department coverage
roster = raw.loc[raw.PersonId.notna() & raw.InstitutionId.isin(institutions)
                 & raw.Year.between(FIRST, LAST) & raw.Rank.isin(RANKS)].copy()
department_years = roster.groupby('DepartmentId').Year.nunique()
roster = roster.loc[roster.DepartmentId.isin(department_years[department_years.ge(CONFIG['min_department_years'])].index)].copy()

various = roster.TaxonomyLevel01Name.str.lower().str.endswith('various', na=False)
has_specific = (~various).groupby(roster.DepartmentId).transform('any')
roster = roster.loc[~various | ~has_specific].copy()

# prefer the newest employment when a researcher appears at multiple institutions in a year
job_start = roster.groupby(['PersonId', 'InstitutionId']).Year.transform('min')
newest_start = job_start.groupby([roster.PersonId, roster.Year]).transform('max')
roster = roster.loc[job_start.eq(newest_start)].drop_duplicates(
    ['PersonId', 'Year', 'InstitutionId', 'DepartmentId', 'TaxonomyLevel01Id'])

In [ ]:
# infer hire before restricting fields; require the preceding census year to be observed
first_seen = roster.groupby('PersonId').Year.transform('min')
hire = roster.loc[roster.Year.eq(first_seen) & roster.Rank.eq('Assistant Professor')
                  & roster.Year.gt(FIRST) & (roster.Year-1).isin(roster.Year.unique())].copy()
hire['field_id'] = hire.TaxonomyLevel01Id.map(FIELD_IDS)
memberships = (hire.dropna(subset=['field_id'])[['PersonId','Year','field_id']]
               .drop_duplicates().rename(columns={'Year':'hire_year'}))

In [ ]:
memberships.to_csv(OUTPUT/'hire_fields.csv', index=False)
# pass publication assets to the next task
for name in ['articles.csv', 'conference-proceedings.csv']:
    link = OUTPUT/name
    if not link.is_symlink():
        link.symlink_to(Path('../input')/name)
display(memberships.groupby('field_id').PersonId.nunique().rename('researchers').reset_index())